# nb_12_gold_kpi_metrics

**Layer**: Gold | **Source**: `lh_silver_insurance` | **Target**: `lh_gold_insurance.gold_kpi_metrics`

**Purpose**: Compute top-level insurance KPIs for executive dashboards and Power BI reports.

**Output Columns**:
| Column | Description |
|---|---|
| kpi_name | Name of the metric |
| kpi_value | Numeric value |
| kpi_unit | Unit (count, currency, percentage, days) |
| kpi_category | Grouping (portfolio, revenue, claims, operations) |

**KPIs Computed**:
- Total Active Policies, Total Customers, Total Agents
- Total Premium Revenue, Avg Annual Premium, Collection Rate
- Total Claims Filed, Open Claims, Approval Rate, Avg Claim Amount
- Loss Ratio, Avg Days to File Claim

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, round as _round,
    when, datediff, current_timestamp
)

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_12_gold_kpi_metrics").getOrCreate()

print("nb_12_gold_kpi_metrics - Gold Layer KPI Computation")

In [ ]:
# ============================================================
# Step 1: Read Silver tables
# ============================================================
df_customers = spark.table("customers")
df_agents = spark.table("agents")
df_policies = spark.table("policies")
df_premiums = spark.table("premiums")
df_claims = spark.table("claims")
df_claim_payments = spark.table("claim_payments")

print("All Silver tables loaded.")

In [ ]:
# ============================================================
# Step 2: Compute KPIs
# ============================================================
kpis = []

# --- Portfolio KPIs ---
total_customers = df_customers.count()
kpis.append(("Total Customers", float(total_customers), "count", "portfolio"))

total_agents = df_agents.count()
active_agents = df_agents.filter(col("status") == "active").count()
kpis.append(("Total Agents", float(total_agents), "count", "portfolio"))
kpis.append(("Active Agents", float(active_agents), "count", "portfolio"))

total_policies = df_policies.count()
active_policies = df_policies.filter(col("status") == "active").count()
kpis.append(("Total Policies", float(total_policies), "count", "portfolio"))
kpis.append(("Active Policies", float(active_policies), "count", "portfolio"))

# --- Revenue KPIs ---
total_premium_due = df_premiums.agg(_sum("amount_due")).collect()[0][0] or 0.0
total_premium_paid = df_premiums.agg(_sum("amount_paid")).collect()[0][0] or 0.0
avg_annual_premium = df_policies.agg(avg("annual_premium")).collect()[0][0] or 0.0
collection_rate = (total_premium_paid / total_premium_due * 100) if total_premium_due > 0 else 0.0

kpis.append(("Total Premium Revenue", round(total_premium_paid, 2), "currency", "revenue"))
kpis.append(("Total Premium Due", round(total_premium_due, 2), "currency", "revenue"))
kpis.append(("Avg Annual Premium", round(avg_annual_premium, 2), "currency", "revenue"))
kpis.append(("Collection Rate", round(collection_rate, 1), "percentage", "revenue"))

# --- Claims KPIs ---
total_claims = df_claims.count()
open_claims = df_claims.filter(col("claim_status").isin("open", "under_review")).count()
approved_claims = df_claims.filter(col("claim_status") == "approved").count()
denied_claims = df_claims.filter(col("claim_status") == "denied").count()
approval_rate = (approved_claims / total_claims * 100) if total_claims > 0 else 0.0
avg_claim_amount = df_claims.agg(avg("estimated_amount")).collect()[0][0] or 0.0
total_claim_payments = df_claim_payments.agg(_sum("payment_amount")).collect()[0][0] or 0.0

kpis.append(("Total Claims Filed", float(total_claims), "count", "claims"))
kpis.append(("Open Claims", float(open_claims), "count", "claims"))
kpis.append(("Approved Claims", float(approved_claims), "count", "claims"))
kpis.append(("Denied Claims", float(denied_claims), "count", "claims"))
kpis.append(("Claim Approval Rate", round(approval_rate, 1), "percentage", "claims"))
kpis.append(("Avg Claim Amount", round(avg_claim_amount, 2), "currency", "claims"))
kpis.append(("Total Claim Payments", round(total_claim_payments, 2), "currency", "claims"))

# --- Operations KPIs ---
# Loss Ratio = Total Claim Payments / Total Premium Revenue
loss_ratio = (total_claim_payments / total_premium_paid * 100) if total_premium_paid > 0 else 0.0
kpis.append(("Loss Ratio", round(loss_ratio, 1), "percentage", "operations"))

# Avg days from loss to filing
avg_days_row = df_claims.withColumn(
    "days_to_file", datediff(col("date_filed"), col("date_of_loss"))
).agg(avg("days_to_file")).collect()[0][0] or 0.0
kpis.append(("Avg Days to File Claim", round(avg_days_row, 1), "days", "operations"))

# Policies per agent
policies_per_agent = total_policies / active_agents if active_agents > 0 else 0.0
kpis.append(("Policies per Active Agent", round(policies_per_agent, 1), "ratio", "operations"))

print(f"Computed {len(kpis)} KPIs")

In [ ]:
# ============================================================
# Step 3: Create and write Gold KPI table
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("kpi_name", StringType(), False),
    StructField("kpi_value", DoubleType(), False),
    StructField("kpi_unit", StringType(), False),
    StructField("kpi_category", StringType(), False)
])

df_gold = spark.createDataFrame(kpis, schema) \
    .withColumn("_processed_at", current_timestamp())

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_kpi_metrics")

gold_count = df_gold.count()
print(f"✓ {gold_count} KPIs written to gold_kpi_metrics")

In [ ]:
# ============================================================
# Validation: Display all KPIs
# ============================================================
print("\n" + "=" * 70)
print("INSURANCE PLATFORM - KEY PERFORMANCE INDICATORS")
print("=" * 70)

for category in ["portfolio", "revenue", "claims", "operations"]:
    print(f"\n  [{category.upper()}]")
    rows = spark.table("gold_kpi_metrics") \
        .filter(col("kpi_category") == category) \
        .select("kpi_name", "kpi_value", "kpi_unit") \
        .collect()
    for row in rows:
        unit = row["kpi_unit"]
        value = row["kpi_value"]
        if unit == "currency":
            formatted = f"${value:,.2f}"
        elif unit == "percentage":
            formatted = f"{value:.1f}%"
        elif unit == "count":
            formatted = f"{int(value):,}"
        else:
            formatted = f"{value:.1f} {unit}"
        print(f"    {row['kpi_name']:<30} {formatted:>15}")

print("\n" + "=" * 70)
print("✓ All KPIs verified.")